元数据与ID映射，数据预处理流水线，包含：数据清洗 → 编码映射 → POI 信息统计 → 时序数据切分
filter_data：清洗原始数据 → 过滤低质数据 → 地理 / 时间 / ID 编码 → 保存标准数据
poi_info：统计每个 POI 的属性和访问时段，生成 POI 信息表
split_data：按时间切分训练 / 验证 / 测试集，保证数据合规
核心价值
把杂乱的原始 TXT 数据 → 标准化 CSV 数据      过滤无效数据，提升模型质量       生成 ID 映射、POI 画像、时序数据集，直接用于 POI 推荐 / 用户轨迹预测模型

In [1]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import csv
from datetime import datetime, timezone, timedelta
from openlocationcode import openlocationcode as olc # 生成PlusCode地理编码
import os
import json
import random
import numpy as np
from ast import literal_eval

In [2]:
# ==================== 项目全局路径配置 ====================
# 已经修改
PROJECT_ROOT = "/home/mysjz/mywork/V2-SID"                     # ← 你的 V2 根目录（根据实际情况调整）
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")         # 数据统一根目录

# 当前处理的数据集（可手动改，或通过参数传入）
#DATAFOLD = 'NYC'                                           # 可改为 'CA' 或 'TKY'
DATAFOLD='NOLA'
# stage（meta、sft 等）
stage = "meta"          #映射文件子目录

# 打印确认
print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据根目录: {DATA_ROOT}")
print(f"当前数据集: {DATAFOLD}")
print(f"输出将保存在: {os.path.join(DATA_ROOT, DATAFOLD)}")

项目根目录: /home/mysjz/mywork/V2-SID
数据根目录: /home/mysjz/mywork/V2-SID/data
当前数据集: NOLA
输出将保存在: /home/mysjz/mywork/V2-SID/data/NOLA


In [3]:
def get_pluscode(latitude, longitude):
    """
    经纬度转换为PlusCode 短码（全球通用的简易地理编码，替代经纬度表示位置）
    给每一条签到数据打上区域标签，方便后续按区域分析
    """
    # 如果纬度/经度为空值，返回无效标记
    if pd.isna(latitude) or pd.isna(longitude):
        return "INVALID"
    try:
        # 把经纬度转为浮点数，并用olc库生成PlusCode
        code = olc.encode(float(latitude), float(longitude))
        # 只返回前6位短码（简化区域表示）
        return code[:6]
    except:
        # 转换失败（非法经纬度）返回无效
        return "INVALID"

def format_time(row):
    """
    清洗原始时间字符串，转为标准的年-月-日 时:分格式，并处理时区
    统一时间格式，解决原始时间杂乱、带时区的问题
    """
    time_str = row['time']
    parts = time_str.split()

    # 拼接需要的时间部分：月 日 时:分:秒 年
    # 把字符串转为datetime对象（无时区）
    clean_time_str = f"{parts[1]} {parts[2]} {parts[3]} {parts[5]}"
    naive_dt = datetime.strptime(clean_time_str, "%b %d %H:%M:%S %Y")
    
    tz = timezone(timedelta(minutes=int(row['tz_offset'])))
    localized_dt = naive_dt.replace(tzinfo=tz)
    return localized_dt.strftime("%Y-%m-%d %H:%M")

def save_mapping(mapping, file_path):
    """
    将原始 ID → 新数字 ID的映射关系保存为 CSV 文件
    记录 ID 转换规则，方便后续溯源原始数据
    """
    with open(file_path, "w", newline="") as uidfile:
        writer = csv.writer(uidfile)
        writer.writerow(["original_uid", "new_uid"])
        for original_uid, new_uid in mapping.items():
            writer.writerow([original_uid, new_uid])


def filter_data(datafold=None, min_user_interactions=1, min_poi_interactions=1): # ⚠️ 阈值改为1，防止二次误伤
    """
    【安全修改版】：带行数监控的特征护送管线
    """
    if datafold is None:
        datafold = DATAFOLD

    dataset_dir = os.path.join(DATA_ROOT, datafold)
    os.makedirs(dataset_dir, exist_ok=True)

    # 【修复 1】：必须严格区分输入和输出文件！
    # 请手动把你带情感的CSV重命名为 NOLA_with_sentiment.csv 放进去
    input_file = os.path.join(dataset_dir, f"{datafold}_with_sentiment.csv")
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"找不到输入文件：{input_file} \n请将你跑完 SAAA2 的文件改名为此名称！")

    print(f"读取带情感特征的全局大表：{input_file}")

    df = pd.read_csv(input_file)
    print(f"初始读取数据量：{len(df)} 行")

    df.rename(columns={'UId': 'uid', 'PId': 'pid', 'Category': 'category',
                       'Latitude': 'latitude', 'Longitude': 'longitude', 'Time': 'time'}, inplace=True)

    # ===================== 数据过滤 =====================
    df['PoiFreq'] = df.groupby('pid')['uid'].transform('count')
    df = df[df['PoiFreq'] >= min_poi_interactions]
    print(f"POI 过滤后数据量：{len(df)} 行")

    df['UserFreq'] = df.groupby('uid')['pid'].transform('count')
    df = df[df['UserFreq'] >= min_user_interactions]
    print(f"User 过滤后数据量：{len(df)} 行")

    df = df.drop(columns=['PoiFreq', 'UserFreq'])

    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df = df.dropna(subset=['latitude', 'longitude'])
    print(f"经纬度清洗后数据量：{len(df)} 行")

    # ===================== 数据增强 =====================
    print("正在生成 PlusCode 区域编码...")
    df["region"] = df.apply(lambda row: get_pluscode(row['latitude'], row['longitude']), axis=1)

    df['time'] = pd.to_datetime(df['time']).dt.strftime("%Y-%m-%d %H:%M")
    df.rename(columns={'time': 'formatted_time'}, inplace=True)

    # ===================== ID重编码 =====================
    print("正在进行 ID 重编码...")
    uids = list(df["uid"].unique())
    pids = list(df["pid"].unique())
    cats = list(df["category"].unique())
    regs = list(df["region"].unique())

    random.shuffle(uids)
    random.shuffle(pids)
    random.shuffle(cats)
    random.shuffle(regs)

    uid_map = {uid: i for i, uid in enumerate(uids)}
    pid_map = {pid: i for i, pid in enumerate(pids)}
    cat_map = {cat: i for i, cat in enumerate(cats)}
    reg_map = {reg: i for i, reg in enumerate(regs)}

    df["new_uid"] = df["uid"].map(uid_map)
    df["new_pid"] = df["pid"].map(pid_map)
    df["new_cid"] = df["category"].map(cat_map)
    df["new_region"] = df["region"].map(reg_map)

    # ===================== 保存结果 =====================
    output_file = os.path.join(dataset_dir, f"{datafold}.csv")

    columns_to_save = [
        'new_uid', 'new_pid', 'new_cid', 'category', 'new_region', 'latitude', 'longitude', 'formatted_time',
        'service', 'environment', 'price', 'location', 'core_experience'
    ]

    final_header = [
        'uid', 'pid', 'cid', 'category', 'region', 'latitude', 'longitude', 'time',
        'service', 'environment', 'price', 'location', 'core_experience'
    ]

    df[columns_to_save].to_csv(output_file, index=False, header=final_header)
    print(f"成功生成带情感分数的 SID 输入文件：{output_file} (最终 {len(df)} 行)")

    # mapping 保存目录
    mapping_dir = os.path.join(dataset_dir, stage)
    os.makedirs(mapping_dir, exist_ok=True)

    save_mapping(uid_map, os.path.join(mapping_dir, "uidmap.csv"))
    save_mapping(pid_map, os.path.join(mapping_dir, "pidmap.csv"))
    save_mapping(cat_map, os.path.join(mapping_dir, "cidmap.csv"))

    print("filter_data 处理完全闭环！去检查 data/NOLA/ 目录吧！")

# ⚠️ 注意这里我把传入参数改为了 1，防止二次过滤误杀数据
datafold = DATAFOLD
filter_data(datafold, min_user_interactions=1, min_poi_interactions=1)

读取带情感特征的全局大表：/home/mysjz/mywork/V2-SID/data/NOLA/NOLA_with_sentiment.csv
初始读取数据量：33852 行
POI 过滤后数据量：33852 行
User 过滤后数据量：33852 行
经纬度清洗后数据量：33852 行
正在生成 PlusCode 区域编码...
正在进行 ID 重编码...
成功生成带情感分数的 SID 输入文件：/home/mysjz/mywork/V2-SID/data/NOLA/NOLA.csv (最终 33852 行)
filter_data 处理完全闭环！去检查 data/NOLA/ 目录吧！


In [4]:
def poi_info(datafold):
    """
    为每个 POI 生成基础信息 + 访问时段统计
    生成POI 画像，知道每个地点的类别、位置、最热门的访问时间
    """
    if datafold is None:
        datafold = DATAFOLD
    # 读取上一步生成的清洗后数据
    dataset_dir = os.path.join(DATA_ROOT, datafold)
    file_path = os.path.join(dataset_dir, f"{datafold}.csv")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"错误：{file_path} 文件未找到。请先运行 filter_data() 生成。")
    
    df = pd.read_csv(file_path)

    # 时间转为标准格式，删除空值
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])

    poi_info_data = []

    # 按POI分组，统计每个POI的信息
    for pid, group in df.groupby('pid'):
        row0 = group.iloc[0]
        category = row0['category']
        region = row0['region']
        latitude = row0['latitude']
        longitude = row0['longitude']
        # 统计该POI在每个小时的访问次数
        hours = group['time'].dt.hour
        hour_counts = hours.value_counts().to_dict()  # {hour: count}

        # 只保留访问次数≥1的时段
        count_threshold = 1
        filtered_hour_counts = {int(h): int(c) for h, c in hour_counts.items() if c >= count_threshold}

        # 按访问次数降序排序
        sorted_hour_counts = dict(
            sorted(filtered_hour_counts.items(), key=lambda x: x[1], reverse=True)
        )

        poi_info_data.append({
            'pid': pid,
            'category': category,
            'region': region,
            'latitude': latitude,
            'longitude': longitude,
            'visit_time_and_count': sorted_hour_counts
        })

    # 保存为POI信息表
    poi_info_df = pd.DataFrame(poi_info_data)
    output_path = os.path.join(dataset_dir, "poi_info.csv")
    poi_info_df.to_csv(output_path, index=False)
    
    print(f"成功创建 {output_path}，共 {len(poi_info_df)} 个 POI")

datafold = DATAFOLD
poi_info(datafold)

成功创建 /home/mysjz/mywork/V2-SID/data/NOLA/poi_info.csv，共 1088 个 POI


In [5]:
# 数据切分，按时间顺序切分训练集 / 验证集 / 测试集，保证测试集无新用户 / 新 POI
#合规切分时序数据，避免模型在训练时看到未来数据，保证评估真实有效
def split_data(datafold, train_ratio=0.8, valid_ratio=0.1, test_ratio=0.1):
    if datafold is None:
        datafold = DATAFOLD
    
    dataset_dir = os.path.join(DATA_ROOT, datafold)
    file_name = os.path.join(dataset_dir, f"{datafold}.csv")
    
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"错误：{file_name} 未找到。请先运行 filter_data()")

    df = pd.read_csv(file_name)

    # 【核心修改 3】：切分数据时，坚决保留我们珍贵的情感分数！
    df = df[['uid', 'pid', 'time', 'service', 'environment', 'price', 'location', 'core_experience']]

    # 按时间排序（签到数据必须按时间切分，模拟真实场景）
    df = df.sort_values(by='time')

    # 按比例计算数据集大小
    train_size = int(train_ratio * len(df))
    valid_size = int(valid_ratio * len(df))

    # 切分数据
    train_df = df[:train_size]
    valid_df = df[train_size:train_size + valid_size]
    test_df = df[train_size + valid_size:]

    ## 关键函数：测试集/验证集只保留训练集中出现过的用户和POI
    def remove_users_pois_test(df_train, df_test):
        users_train = df_train['uid'].unique()
        pois_train = df_train['pid'].unique()
        df_test = df_test[df_test['uid'].isin(users_train)]
        df_test = df_test[df_test['pid'].isin(pois_train)]
        return df_test

    # ===================== 保存切分结果 =====================
    train_path = os.path.join(dataset_dir, "train_data.csv")
    valid_path = os.path.join(dataset_dir, "valid_data.csv")
    test_path  = os.path.join(dataset_dir, "test_data.csv")

    # 所有的 to_csv 都会带着那 5 列情感一起保存
    train_df.to_csv(train_path, index=False)
    print(f"保存带情感的训练集：{train_path}")

    # 过滤验证集，保留该用户的全部历史数据（方便模型学习用户偏好）
    valid_df = remove_users_pois_test(train_df, valid_df)
    expanded_valid_df = df[df['uid'].isin(valid_df['uid'].unique())]
    expanded_valid_df.to_csv(valid_path, index=False)
    print(f"保存带情感的验证集：{valid_path}")

    # 过滤测试集，同上
    test_df = remove_users_pois_test(train_df, test_df)
    expanded_test_df = df[df['uid'].isin(test_df['uid'].unique())]
    expanded_test_df.to_csv(test_path, index=False)
    print(f"保存带情感的测试集：{test_path}")
    print("时序切分完成！情感分流成功！")

datafold = DATAFOLD
split_data(datafold)

保存带情感的训练集：/home/mysjz/mywork/V2-SID/data/NOLA/train_data.csv
保存带情感的验证集：/home/mysjz/mywork/V2-SID/data/NOLA/valid_data.csv
保存带情感的测试集：/home/mysjz/mywork/V2-SID/data/NOLA/test_data.csv
时序切分完成！情感分流成功！
